# Multi-Agent Spend Control — FiGuard + Langfuse OTEL Tracing

[![PyPI](https://img.shields.io/pypi/v/figuard.svg)](https://pypi.org/project/figuard/)
[![GitHub](https://img.shields.io/badge/github-figuard%2Ffiguard--core-blue)](https://github.com/figuard/figuard-core)

This notebook shows how FiGuard authorization decisions appear as OTEL spans inside a Langfuse trace.

**Scenario:** A travel booking fleet with an orchestrator and two specialist sub-agents.
- Orchestrator creates a shared fleet budget ($1,000) and issues delegation tokens
- `flight-booker` gets a $600 cap — books two flights successfully
- `hotel-booker` gets a $500 cap — tries a $550 hotel (denied), then books $420

In Langfuse, each FiGuard authorization appears as a child span under the tool call that triggered it.
The denied hotel booking shows as an ERROR span — in the same trace as the successful bookings.

**No LLM calls, no API keys required** — uses the shared public sandbox automatically.

> ☁️ **This notebook runs against the free shared sandbox** — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")`. It uses features that need a server (fleet delegation, category allocations, shadow mode, cross-process budgets, ledger/void-tree). The sandbox is for evaluation only — **not for production** (may be wiped at any time; no SLA).
>
> **Three ways to run FiGuard — same API, same rules. Swap only the client constructor:**
>
> - ☁️ **Free sandbox** _(this notebook)_ — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")` — quick hosted evaluation.
> - 🏢 **Self-hosted server** — `FiGuardClient(base_url="https://your-figuard", api_key="...")` — your data, your infra; the production path for the server features this notebook uses. [Self-host →](https://github.com/figuard/figuard-core#self-hosting)
> - 💻 **Embedded** — `FiGuardClient()` — in-process, zero infra (single process; the server-only features used here aren't available embedded).

## Step 1 — Install dependencies

In [ ]:
%pip install -qU figuard[langchain,opentelemetry] langfuse opentelemetry-exporter-otlp-proto-http

## Quick start (v1.1.0+)

`auto_guard_langchain` works with OTEL out of the box — no extra wiring needed:

```python
from figuard import auto_guard_langchain
executor = auto_guard_langchain(executor, budget=500, velocity_max_per_minute=10)
# FiGuard authorization spans appear in Langfuse automatically when OTEL is configured
```

This notebook shows the **full OTEL setup** — wiring Langfuse as the OTEL backend so authorization decisions appear as `GUARDRAIL` spans alongside your LLM traces. The approach works with any OTEL-compatible backend (Jaeger, Honeycomb, Datadog, etc.).

## Step 2 — Configure Langfuse credentials

Get your public and secret keys from [cloud.langfuse.com](https://cloud.langfuse.com) under **Settings → API Keys**.

In [ ]:
import os
import getpass

if not os.environ.get("LANGFUSE_PUBLIC_KEY"):
    os.environ["LANGFUSE_PUBLIC_KEY"] = getpass.getpass("Langfuse public key (pk-lf-...): ")
if not os.environ.get("LANGFUSE_SECRET_KEY"):
    os.environ["LANGFUSE_SECRET_KEY"] = getpass.getpass("Langfuse secret key (sk-lf-...): ")

# US region default — change to https://eu.cloud.langfuse.com if your account is in EU
os.environ.setdefault("LANGFUSE_HOST", "https://us.cloud.langfuse.com")
os.environ["FIGUARD_SUPPRESS_SANDBOX_WARNING"] = "1"

print("Credentials set.")

## Step 3 — Wire OTEL to Langfuse

FiGuard picks up the global OTEL provider automatically — no extra config needed.
Every `authorize()`, `confirm_event()`, `fail_event()`, and `void_event()` call emits a span.

In [ ]:
from langfuse import Langfuse

lf = Langfuse(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    host=os.environ["LANGFUSE_HOST"],
)

assert lf.auth_check(), "Langfuse auth failed — check your keys and host"
print(f"Connected to Langfuse at {os.environ['LANGFUSE_HOST']}")
print("Authorization denials will appear as GUARDRAIL observations in each trace.")

## Step 4 — Create fleet budget and delegation tokens

The orchestrator owns the fleet budget. Sub-agents never see the fleet session token — they only
receive their own delegation token with a hard per-agent cap.

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")  # shared public sandbox

# Fleet budget — shared $1,000 ceiling
fleet = client.create_budget(
    user_id="travel_orchestrator",
    total_limit=1_000.00,
    currency="USD",
    expires_in="24h",  # flat budget — delegation token caps enforce per-agent limits
)
fleet_token = fleet.primary_token.session_token

print(f"Fleet budget:  {fleet.id}")
print(f"Total limit:   ${fleet.total_limit:,.2f}  |  Available: ${fleet.available_quantity:,.2f}")
print(f"(Per-agent caps enforced via delegation tokens — not fleet allocations)")

# Delegation token: flight-booker — $600 cap
flight_token = client.create_delegation_token(
    budget_id=fleet.id,
    label="flight-booker",
    caps=[{"category": "flights", "limit": 600.00}],
)

# Delegation token: hotel-booker — $500 cap
hotel_token = client.create_delegation_token(
    budget_id=fleet.id,
    label="hotel-booker",
    caps=[{"category": "hotels", "limit": 500.00}],
)

print(f"\nflight-booker token: {flight_token.session_token[:20]}...  cap: $600")
print(f"hotel-booker  token: {hotel_token.session_token[:20]}...  cap: $500")

## Step 5 — Run the booking fleet

Each sub-agent runs inside a root OTEL span opened by the orchestrator.
FiGuard emits a `figuard.authorize` child span for every tool call.

Watch what happens:
- Flight 1 ($280) and Flight 2 ($290): **AUTHORIZED**
- Hotel attempt ($550): **DENIED** — `DELEGATE_CAP_EXCEEDED`, cap is $500
- Hotel retry ($420): **AUTHORIZED**

The denied booking shows as an ERROR span in Langfuse — nested inside the same trace.

In [ ]:
from uuid import uuid4
from langchain_core.tools import ToolException
from figuard.integrations.langchain import FiGuardCallbackHandler

flight_h = FiGuardCallbackHandler(
    client=client,
    session_token=flight_token.session_token,
    agent_id="flight-booker",
    tool_category_map={"book_flight": "flights"},
)
hotel_h = FiGuardCallbackHandler(
    client=client,
    session_token=hotel_token.session_token,
    agent_id="hotel-booker",
    tool_category_map={"book_hotel": "hotels"},
)

print("Running travel booking fleet...\n")

with lf.start_as_current_observation(
    as_type="agent",
    name="travel-booking-orchestrator",
    input={"task": "Book NYC-LAX round trip with hotel", "fleet_budget": 1000.00},
) as root:

    print("--- flight-booker (delegation cap: $600) ---")
    with lf.start_as_current_observation(as_type="agent", name="flight-booker",
                                         metadata={"delegation_cap": 600.00}) as fa:
        r = uuid4()
        flight_h.on_tool_start({"name": "book_flight"},
                                '{"amount": 280.0, "description": "NYC to LAX"}', run_id=r)
        flight_h.on_tool_end("NYC to LAX booked", run_id=r)
        print("  [OK]  book_flight   $280.00   NYC to LAX        AUTHORIZED")

        r = uuid4()
        flight_h.on_tool_start({"name": "book_flight"},
                                '{"amount": 290.0, "description": "LAX to NYC return"}', run_id=r)
        flight_h.on_tool_end("LAX to NYC return booked", run_id=r)
        print("  [OK]  book_flight   $290.00   LAX to NYC        AUTHORIZED")
        fa.update(output={"flights_booked": 2, "total": 570.0})

    print("\n--- hotel-booker (delegation cap: $500) ---")
    with lf.start_as_current_observation(as_type="agent", name="hotel-booker",
                                         metadata={"delegation_cap": 500.00}) as ha:
        r = uuid4()
        try:
            hotel_h.on_tool_start({"name": "book_hotel"},
                                   '{"amount": 550.0, "description": "Grand Hyatt 3 nights"}',
                                   run_id=r)
        except ToolException:
            print("  [X]   book_hotel    $550.00   Grand Hyatt        DENIED: DELEGATE_CAP_EXCEEDED")
            with lf.start_as_current_observation(
                as_type="guardrail",
                name="figuard.deny",
                level="ERROR",
                metadata={
                    "denial_reason": "DELEGATE_CAP_EXCEEDED",
                    "requested": 550.0,
                    "delegation_cap": 500.0,
                    "agent": "hotel-booker",
                },
            ):
                pass

        r = uuid4()
        hotel_h.on_tool_start({"name": "book_hotel"},
                               '{"amount": 420.0, "description": "Holiday Inn 3 nights"}',
                               run_id=r)
        hotel_h.on_tool_end("Holiday Inn booked", run_id=r)
        print("  [OK]  book_hotel    $420.00   Holiday Inn        AUTHORIZED")
        ha.update(output={"hotel_booked": True, "amount": 420.0})

    root.update(output={"status": "completed", "total_spent": 990.0})

lf.flush()
print("\nDone. Open Langfuse -> Tracing and look for: travel-booking-orchestrator")

## Step 6 — What you see in Langfuse

Open **cloud.langfuse.com → Traces**. Find the `travel-booking-orchestrator` trace.

You should see:
```
[travel-booking-orchestrator]               ← root span (orchestrator)
  ├── [figuard.authorize]  OK   $280        ← flight 1 authorized
  ├── [figuard.confirm]    OK   $280        ← flight 1 confirmed
  ├── [figuard.authorize]  OK   $290        ← flight 2 authorized
  ├── [figuard.confirm]    OK   $290        ← flight 2 confirmed
  ├── [figuard.authorize]  ERR  $550        ← hotel denied (ERROR span)
  │     error: DELEGATE_CAP_EXCEEDED
  ├── [figuard.authorize]  OK   $420        ← hotel retry authorized
  └── [figuard.confirm]    OK   $420        ← hotel confirmed
```

Each span carries attributes:
- `figuard.agent_id` — which sub-agent made the call
- `figuard.requested_quantity` — what was requested
- `figuard.claimed_category` — which allocation was targeted
- `figuard.decision` — AUTHORIZED or DENIED
- `figuard.denial_reason` — why it was denied (on ERROR spans)

The denied booking is an ERROR span — not a silent failure, not a separate log to check.
Authorization and execution are in the same trace.

## Step 7 — Inspect the FiGuard spend ledger

The Langfuse trace shows the execution view. The FiGuard ledger shows the authorization view.
Both correlated by the same `trace_id`.

In [ ]:
page = client.get_ledger(fleet.id, page=0, size=20)

print(f"Fleet budget: {fleet.id}")
print(f"Total events: {page.total_elements}")
print()
print(f"  {'DECISION':<16}  {'AMOUNT':>8}  {'AGENT':<16}  DENIAL")
print(f"  {'-'*16}  {'-'*8}  {'-'*16}  {'-'*30}")

for ev in page.events:
    icon   = "[OK] " if ev.decision in ("AUTHORIZED", "CONFIRMED") else "[X]  "
    amount = ev.confirmed_quantity or ev.requested_quantity or 0
    denial = ev.denial_reason or ""
    print(f"  {icon}{ev.decision:<14}  ${amount:>7.2f}  {ev.agent_id:<16}  {denial}")

## Step 8 — Final budget state

In [ ]:
final = client.get_budget(fleet.id)

print(f"Fleet budget summary")
print(f"  Total limit:   ${final.total_limit:,.2f}")
print(f"  Spent:         ${final.quantity_spent:,.2f}")
print(f"  Reserved:      ${final.quantity_reserved:,.2f}")
print(f"  Available:     ${final.available_quantity:,.2f}")
print()
print("Per-category allocation usage:")
for alloc in final.allocations:
    bar_pct = int((alloc.quantity_spent / alloc.limit) * 20) if alloc.limit else 0
    bar = "[" + "#" * bar_pct + "-" * (20 - bar_pct) + "]"
    print(f"  {alloc.category:<10} {bar}  "
          f"${alloc.quantity_spent:,.2f} / ${alloc.limit:,.2f}")

## What this demonstrates

| Capability | Where it shows up |
|---|---|
| Per-agent delegation caps | hotel-booker denied at $550, fleet still had $710 remaining |
| Two-level enforcement | delegation cap AND fleet ceiling both enforced independently |
| Authorization as OTEL span | every `authorize()` is a child span — visible in Langfuse |
| Denied = ERROR span | the $550 denial shows as an ERROR span, not a silent failure |
| Append-only ledger | every event recorded — authorized, denied, confirmed |
| Zero-config sandbox | ran with no API key — shared sandbox connects automatically |

To run against your own FiGuard instance, set `FIGUARD_API_KEY` and `FIGUARD_BASE_URL`
before importing `FiGuardClient`.

Self-hosting: [github.com/figuard/figuard-core](https://github.com/figuard/figuard-core)
```bash
docker compose up -d
```